# Chạy AI service trích xuất CV trên Colab GPU (llama.cpp + CUDA)

Notebook này chỉ để **chạy API** (`POST /api/v1/extract-cv`) bằng một file GGUF đã train, trên GPU T4, rồi mở Cloudflare tunnel để gọi từ máy local / Claude.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU** → chọn `MODEL` ở cell [1] → **Runtime → Run all**.
Mất ~3–5 phút nếu có bản llama.cpp dựng sẵn có CUDA; ~15–20 phút nếu phải tự biên dịch (cell [3] tự xử lý).

Khác các notebook cũ:
- llama.cpp **bắt buộc chạy GPU**: không có CUDA thì dừng với thông báo, không âm thầm chạy CPU (CPU làm mỗi CV mất > 100 s và Cloudflare cắt kết nối, lỗi 524);
- đặt sẵn `MODEL_ENGINE=llama_cpp` (config mặc định của service là `transformers`);
- đo tốc độ bằng một CV hư cấu trước khi mở tunnel.

Không gửi CV thật qua tunnel công khai nếu không cần; service không ghi nội dung CV vào log.

In [ ]:
# [1] Cấu hình
MODEL = "v6.1"   # "v6.1" | "v4" | "v3" | "base" (Qwen3-1.7B gốc, chưa fine-tune)
REPO, BRANCH = "https://github.com/long20102004/recruit_system.git", "dev_hocnv"
SERVICE_DIR = "/content/recruit_system/ai-service/extraction"

# File GGUF Q4_K_M của từng model (link Drive công khai của dự án / Hugging Face)
GGUF_SOURCES = {
    "v6.1": "drive:1lBbj4we25DSszqzuLSsilwzzp8rMSBUU",   # finetune_output_v6.1/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
    "v4":   "drive:1mKd0SmhnUKpJTXkH3xQs0puEwYgqFQWn",   # finetune_output_v4/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
    "v3":   "drive:1mqmolAF-4nJXg1db89oHAYZKYlZ3CUcq",   # finetune_output_v3/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
    "base": "url:https://huggingface.co/unsloth/Qwen3-1.7B-GGUF/resolve/main/Qwen3-1.7B-Q4_K_M.gguf",
}
# Muốn dùng file trên Drive của bạn thay cho link trên: đặt đường dẫn đầy đủ, ví dụ
# GGUF_DRIVE_PATH = "/content/drive/MyDrive/cv-analysis-service/finetune_output_v6.1/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf"
GGUF_DRIVE_PATH = ""

PROMPT_VERSION = "v3"   # v3, v4, v6.1 và base (zero-shot) đều dùng prompt v3
assert MODEL in GGUF_SOURCES, f"MODEL phải là một trong {list(GGUF_SOURCES)}"
print("MODEL =", MODEL, "| PROMPT_VERSION =", PROMPT_VERSION)

In [ ]:
# [2] Kiểm tra GPU + lấy code
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "Chưa có GPU: Runtime > Change runtime type > T4 GPU (hoặc đã hết hạn mức GPU của tài khoản)"
print("GPU:", gpu.stdout.strip())

%cd /content
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
assert os.path.isfile(f"{SERVICE_DIR}/main.py"), f"Không thấy {SERVICE_DIR}/main.py: kiểm tra BRANCH"
!git -C /content/recruit_system log --oneline -1
!pip install -q fastapi uvicorn python-dotenv pydantic json-repair rapidfuzz "neo4j>=5.20" gdown

In [ ]:
# [3] llama.cpp có CUDA (bắt buộc)
import glob, site
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
!ldconfig 2>/dev/null
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + ["/usr/local/cuda/lib64", os.environ.get("LD_LIBRARY_PATH", "")])

def gpu_offload_ok():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"],
                       capture_output=True, text=True, env=os.environ)
    return r.stdout.strip().endswith("True")

ok = False
# 1) bản dựng sẵn có CUDA: --only-binary để pip KHÔNG tự biên dịch bản CPU từ mã nguồn
for cu in ("cu124", "cu125", "cu122", "cu121"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
                        "--only-binary=llama-cpp-python", "llama-cpp-python",
                        "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{cu}"],
                       capture_output=True, text=True)
    if r.returncode == 0 and gpu_offload_ok():
        print("llama-cpp-python: bản dựng sẵn", cu, "+ GPU OK"); ok = True; break
    print("Không dùng được bản dựng sẵn", cu)
# 2) không có bản dựng sẵn hợp lệ: biên dịch có CUDA (~15-20 phút)
if not ok:
    print("Biên dịch llama-cpp-python với CUDA (~15-20 phút)...")
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on", FORCE_CMAKE="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
                        "--no-binary=llama-cpp-python", "llama-cpp-python"], env=env, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
    ok = r.returncode == 0 and gpu_offload_ok()
assert ok, "llama-cpp-python KHÔNG có CUDA: dừng lại để không chạy CPU (chậm, lỗi 524). Gửi output cell này cho Claude."
print("llama.cpp dùng GPU: OK")

In [ ]:
# [4] Lấy file GGUF
os.makedirs("/content/models", exist_ok=True)
if GGUF_DRIVE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.isfile(GGUF_DRIVE_PATH), f"Không thấy {GGUF_DRIVE_PATH}"
    GGUF_PATH = f"/content/models/{MODEL}.gguf"
    !cp "{GGUF_DRIVE_PATH}" "{GGUF_PATH}"
else:
    kind, ref = GGUF_SOURCES[MODEL].split(":", 1)
    GGUF_PATH = f"/content/models/{MODEL}.gguf"
    if not os.path.exists(GGUF_PATH):
        cmd = [sys.executable, "-m", "gdown", "-q", ref, "-O", GGUF_PATH] if kind == "drive" else ["wget", "-q", "-O", GGUF_PATH, ref]
        subprocess.run(cmd, check=True)
assert open(GGUF_PATH, "rb").read(4) == b"GGUF", "File tải về không phải GGUF (link sai hoặc chưa chia sẻ công khai)"
print("GGUF:", GGUF_PATH, round(os.path.getsize(GGUF_PATH) / 1e9, 2), "GB")

In [ ]:
# [5] Chạy service + kiểm tra tốc độ bằng CV hư cấu
import json, time, urllib.request
if "srv" in globals():
    srv.kill(); time.sleep(2)
%cd {SERVICE_DIR}
env = dict(os.environ, MODEL_ENGINE="llama_cpp", PROMPT_VERSION=PROMPT_VERSION, GGUF_MODEL_PATH=GGUF_PATH)
for k in ("KB_DSN", "NEO4J_URI"):
    env.pop(k, None)   # Colab không có Postgres / Neo4j: KB đọc từ seed CSV
srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT, env=env)
h = None
for _ in range(150):
    try:
        h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")); break
    except Exception:
        if srv.poll() is not None:
            break
        time.sleep(2)
print("health:", h)
if not (h and h.get("engine") == "llama_cpp"):
    print(open("/content/server.log").read()[-4000:])
    raise RuntimeError("Service chưa chạy đúng: xem log ở trên")

SAMPLE = '''NGUYEN VAN FICTION
fiction.dev@example.com | 0900 000 111 | github.com/fiction-dev
EDUCATION
Fiction University of Technology    2022 - 2026
Information Technology - GPA: 3.4/4
EXPERIENCE
Fiction Software JSC    06/2025 - Present
Backend Intern
- Built REST APIs with Spring Boot and PostgreSQL
PROJECTS
Online Bookstore    02/2025 - 05/2025
- Designed the database and payment API
Tech: Java, Spring Boot, MySQL
SKILLS
Java, Spring Boot, PostgreSQL, Docker, Git
LANGUAGES
English: TOEIC 750'''
req = urllib.request.Request("http://127.0.0.1:8000/api/v1/extract-cv", data=json.dumps({"cv_text": SAMPLE}).encode(),
                             headers={"Content-Type": "application/json"}, method="POST")
t0 = time.time(); out = json.load(urllib.request.urlopen(req, timeout=600)); dt = time.time() - t0
vram = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(f"CV hư cấu: {dt:.1f} s | VRAM đang dùng: {vram}")
print(json.dumps(out["data"], ensure_ascii=False)[:600])
if dt > 30:
    print("CẢNH BÁO: quá chậm so với GPU (thường 3-10 s): kiểm tra cell [3]")

In [ ]:
# [6] Mở Cloudflare tunnel -> URL gửi cho Claude
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
if "tun" in globals():
    tun.kill()
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                       stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
print("=" * 70)
print(f"URL ai-service ({MODEL}, gửi cho Claude):", url)
print("=" * 70)

In [ ]:
# [7] Giữ phiên sống (dừng cell này khi xong: nút Stop)
while True:
    time.sleep(120)
    alive = srv.poll() is None and tun.poll() is None
    try:
        h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=10))
        status = h.get("status")
    except Exception as e:
        status = type(e).__name__
    print(time.strftime("%H:%M:%S"), "| service + tunnel sống:", alive, "| health:", status, flush=True)
    if not alive:
        print(open("/content/server.log").read()[-2000:]); break